# UD03 - SAA: Análisis de Sentimiento

**Autor:** Julio García Ortiz

## Importación de Librerías

In [21]:
import kagglehub
import pandas as pd
import numpy as np
import re
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline

## Obtener datos del Dataset mediante libreria de Kagglehub

In [32]:
# Descargamos el dataset de reseñas de películas IMDB desde Kaggle
path = kagglehub.dataset_download("nick2908/imdb-movie-reviews-for-sentiment-analysis")
print("Dataset descargado en:", path)

# Cargamos el CSV en un DataFrame

files = os.listdir(path)
print("Archivos disponibles:", files)

df = pd.read_csv(f"{path}/{files[0]}")
print(df.head())
print("Columnas:", df.columns.tolist())
print("Shape:", df.shape)

Using Colab cache for faster access to the 'imdb-movie-reviews-for-sentiment-analysis' dataset.
Dataset descargado en: /kaggle/input/imdb-movie-reviews-for-sentiment-analysis
Archivos disponibles: ['finalReviews.csv']
                                              review  label
0  There isn't too much in the way of suspense or...      1
1  I cried and laughed and blah, blah, blah. CGI ...      1
2                      Not as good as infinity war..      1
3  First review from me. This film deserves it. A...      1
4  This film is an emotional rollercoaster with s...      1
Columnas: ['review', 'label']
Shape: (302, 2)


## Comprobación de los datos obtenidos


In [33]:
# Ahora sí renombramos (ajusta si los nombres siguen siendo distintos)
df = df.rename(columns={"review": "texto", "label": "categoria"})

print(df[["texto", "categoria"]].head())
print("\nDistribución de categorías:")
print(df["categoria"].value_counts())

                                               texto  categoria
0  There isn't too much in the way of suspense or...          1
1  I cried and laughed and blah, blah, blah. CGI ...          1
2                      Not as good as infinity war..          1
3  First review from me. This film deserves it. A...          1
4  This film is an emotional rollercoaster with s...          1

Distribución de categorías:
categoria
1    170
0    132
Name: count, dtype: int64


## Separamos el texto en tokens y limpiamos el texto


In [34]:
def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r"<.*?>", "", texto)       # Quitar etiquetas HTML
    texto = re.sub(r"[^a-z\s]", "", texto)    # Quitar caracteres especiales
    texto = texto.strip()
    return texto

df["texto_limpio"] = df["texto"].apply(limpiar_texto)
print(df["texto_limpio"].head())

0    there isnt too much in the way of suspense or ...
1    i cried and laughed and blah blah blah cgi was...
2                          not as good as infinity war
3    first review from me this film deserves it a s...
4    this film is an emotional rollercoaster with s...
Name: texto_limpio, dtype: object


## Selección de datos (Split)

In [35]:
X = df["texto_limpio"]
y = df["categoria"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Tamaño entrenamiento:", X_train.shape)
print("Tamaño prueba:", X_test.shape)

Tamaño entrenamiento: (241,)
Tamaño prueba: (61,)


## Fase de entrenamiento

In [36]:
modelo = Pipeline([
    ("vectorizer", CountVectorizer(stop_words="english")),
    ("clasificador", MultinomialNB())
])

modelo.fit(X_train, y_train)
print("Modelo entrenado correctamente ✅")

Modelo entrenado correctamente ✅


## Fase de validación

In [37]:
y_pred_train = modelo.predict(X_train)
print("Accuracy en entrenamiento:", f"{accuracy_score(y_train, y_pred_train):.2%}")

y_pred = modelo.predict(X_test)
print("Accuracy en test:", f"{accuracy_score(y_test, y_pred):.2%}")

Accuracy en entrenamiento: 93.36%
Accuracy en test: 75.41%


## Predicciones

In [38]:
def predecir_sentimiento(texto_nuevo):
    texto_limpio = limpiar_texto(texto_nuevo)
    prediccion = modelo.predict([texto_limpio])[0]
    return f"Sentimiento: {'Positivo ✅' if prediccion == 1 else 'Negativo ❌'}"

print(predecir_sentimiento("This movie was absolutely amazing! I loved every minute."))
print(predecir_sentimiento("Terrible film, complete waste of time."))

Sentimiento: Positivo ✅
Sentimiento: Negativo ❌


## Optimización

In [39]:
modelo_optimizado = Pipeline([
    ("vectorizer", CountVectorizer(stop_words="english", ngram_range=(1,2))),
    ("tfidf", TfidfTransformer()),
    ("clasificador", MultinomialNB())
])

modelo_optimizado.fit(X_train, y_train)
y_pred_opt = modelo_optimizado.predict(X_test)

print("Accuracy modelo optimizado:", f"{accuracy_score(y_test, y_pred_opt):.2%}")

Accuracy modelo optimizado: 80.33%


In [40]:
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2%}")

print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy Score: 75.41%

--- Confusion Matrix ---
[[20  5]
 [10 26]]


In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorizamos para todos los modelos
cv = CountVectorizer(stop_words="english")
X_train_count = cv.fit_transform(X_train)
X_test_count = cv.transform(X_test)

tfidf = TfidfVectorizer(stop_words="english")
X_train_vec = tfidf.fit_transform(X_train).toarray()
X_test_vec = tfidf.transform(X_test).toarray()

# Multinomial
mnb = MultinomialNB()
mnb.fit(X_train_count, y_train)
acc_mnb = accuracy_score(y_test, mnb.predict(X_test_count))

# Gaussian
gnb = GaussianNB()
gnb.fit(X_train_vec, y_train)
acc_gnb = accuracy_score(y_test, gnb.predict(X_test_vec))

# Bernoulli
bnb = BernoulliNB()
bnb.fit(X_train_count, y_train)
acc_bnb = accuracy_score(y_test, bnb.predict(X_test_count))

# Tabla resumen
resultados = pd.DataFrame({
    "Modelo": ["Multinomial NB", "Gaussian NB", "Bernoulli NB"],
    "Accuracy": [f"{acc_mnb:.2%}", f"{acc_gnb:.2%}", f"{acc_bnb:.2%}"]
})
print(resultados.to_string(index=False))

conclusion = """
CONCLUSIÓN:
- Multinomial NB: mejor para texto, trabaja con frecuencias de palabras.
- Bernoulli NB: bueno cuando solo importa si una palabra aparece o no.
- Gaussian NB: pensado para datos continuos, por eso rinde peor con texto.
"""
print(conclusion)

        Modelo Accuracy
Multinomial NB   75.41%
   Gaussian NB   75.41%
  Bernoulli NB   73.77%

CONCLUSIÓN:
- Multinomial NB: mejor para texto, trabaja con frecuencias de palabras.
- Bernoulli NB: bueno cuando solo importa si una palabra aparece o no.
- Gaussian NB: pensado para datos continuos, por eso rinde peor con texto.

